# TF families and receptor target sets: CollecTRI versus SCENIC+ eRegulons

Notebooks 01 and 02 score the same age contrasts against two networks: literature-curated TF
regulons (CollecTRI) and enhancer-driven regulons inferred from the thymus multiome (SCENIC+). A
TF-by-TF comparison of the two is only partly meaningful, because the two networks assign different
target sets to the same TF; what should agree, if both see the same biology, is the **families** of
TFs they nominate. This short notebook reads the stromal shared rankings written by notebooks 01
and 02 and asks:

1. which TF families populate the top-20 up / down lists of each network;
2. how whole families rank across the two networks, beyond the top-20 cut;
3. for TFs present in both networks, how well the two rankings agree;
4. for **AR and NR3C1**, how the CollecTRI regulon and the SCENIC+ eRegulon overlap, and how shared
   and network-specific targets behave in the stromal age contrasts.

Inputs: `results/*_sharedRanking_stroma.csv`, `results/*_top20_sharedStroma.csv`,
`results/*_scores_perCellType.csv` (from notebooks 01 and 02), the two networks in `data/`, the
DESeq2 tables. Runtime < 2 min.


In [1]:
import os, re, sys, time, warnings
import numpy as np
import pandas as pd
import decoupler as dc
from scipy.stats import norm, spearmanr, percentileofscore
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_rgb
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print(f'python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'decoupler {dc.__version__} | matplotlib {matplotlib.__version__} | seaborn {sns.__version__}')


python 3.12.0 | numpy 2.2.4 | pandas 2.2.3 | decoupler 2.1.1 | matplotlib 3.10.1 | seaborn 0.13.2


In [2]:
# ------------------------------------------------------------------ paths (edit for your setup)
DEA_DIR = '/nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/data/analyses/dea'  # DESeq2 tables
HERE    = os.environ.get('NB_OUT_DIR', os.getcwd())
DATA_DIR, RESULTS, FIGURES = f'{HERE}/data', f'{HERE}/results', f'{HERE}/figures'
for d in (RESULTS, FIGURES):
    os.makedirs(d, exist_ok=True)

LEVEL       = 'taa_l2_v2'
CONTRASTS   = ['paed_vs_infant', 'adult_vs_infant']
PREFIX      = f'thyAgeing_TFfamilies_ageDE_{LEVEL}'
CT_PREFIX   = f'thyAgeing_CollecTRI_ageDE_{LEVEL}'       # outputs of notebook 01
SC_PREFIX   = f'thyAgeing_SCENICplus_ageDE_{LEVEL}'      # outputs of notebook 02
STEROID_TFS = ['AR', 'NR3C1', 'PGR', 'ESR1', 'ESR2']
STROMA_L1   = ['TEC', 'Fb', 'EC', 'Mural']
STAT_CAP    = 20.0
N_TOP       = 20
NET_COLORS  = {'CollecTRI': '#4C72B0', 'SCENIC+': '#DD8452'}
print(f'{LEVEL} | reading notebook 01/02 outputs from {RESULTS}')


taa_l2_v2 | reading notebook 01/02 outputs from /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github/results


In [3]:
# ------------------------------------------------------------------ figure style
import logging
logging.getLogger('fontTools').setLevel(logging.WARNING)   # PDF font subsetting is chatty at INFO
logging.getLogger('matplotlib').setLevel(logging.WARNING)
try:                                                    # Arial if available (Nature house font)
    font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
except Exception:
    pass
rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42, 'ps.fonttype': 42,             # text stays editable in Illustrator
    'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'figure.titlesize': 8,
    'axes.linewidth': 0.5, 'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2, 'xtick.major.pad': 1.5, 'ytick.major.pad': 1.5,
    'savefig.dpi': 300, 'savefig.bbox': 'standard',    # 'standard': the PDF keeps the size in mm below
    'savefig.transparent': False, 'figure.dpi': 110,
})
MM = 1 / 25.4                                            # inches per mm
NATURE_W = {'single': 89.0, 'two-thirds': 120.0, 'double': 183.0}   # Nature figure widths, mm
MISSING = '#e3e0e0'                                      # cell type not tested in a contrast
DIVERGING = sns.color_palette('vlag', as_cmap=True).copy()
DIVERGING.set_bad(MISSING)

# One colour + marker per steroid receptor (identical across all figures of the manuscript).
RECEPTOR_STYLE = {'AR':    ('#d95f02', 'o'),
                  'NR3C1': ('#1b9e77', '*'),
                  'PGR':   ('#7570b3', 'D'),
                  'ESR1':  ('#b2182b', 's'),
                  'ESR2':  ('#56B4E9', '^')}


def mm_fig(width_mm, height_mm):
    """A figure of exactly this size in millimetres; constrained layout keeps labels inside it."""
    return plt.figure(figsize=(width_mm * MM, height_mm * MM), layout='constrained')


def save_fig(fig, name):
    """Save `FIGURES/<name>.pdf` at the figure's size in mm, and report it."""
    path = f'{FIGURES}/{name}.pdf'
    fig.savefig(path)
    w, h = np.asarray(fig.get_size_inches()) * 25.4
    print(f'saved {os.path.basename(path)}  ({w:.0f} x {h:.0f} mm)')
    return path


In [4]:
# ------------------------------------------------------------------ cell-type order and lineage
# `thyAgeing_celltype_levels.csv` is the curated annotation hierarchy of the atlas (one row per
# finest label; columns taa_l1, taa_l2_v2, taa_l3, ... in the atlas' display order). It fixes the
# order of cell types in every figure and defines the lineages: "stroma" = TEC + Fb + EC + Mural.
CT_LEVELS = pd.read_csv(f'{DATA_DIR}/thyAgeing_celltype_levels.csv', dtype=str)
assert LEVEL in CT_LEVELS.columns, f'{LEVEL} is not a column of thyAgeing_celltype_levels.csv'
LINEAGE = (CT_LEVELS[[LEVEL, 'taa_l1']].dropna().drop_duplicates(LEVEL)
           .set_index(LEVEL)['taa_l1'].to_dict())          # cell type -> broad lineage (taa_l1)


def ct_order(lineages=None, present=None, level=LEVEL):
    """Cell types of `level` in atlas order, restricted to taa_l1 `lineages` and/or to `present`."""
    d = CT_LEVELS if lineages is None else CT_LEVELS[CT_LEVELS['taa_l1'].isin(lineages)]
    cts = list(dict.fromkeys(d[level].dropna()))
    return cts if present is None else [c for c in cts if c in set(present)]


# lineage colours: one distinct hue per broad lineage (stromal lineages first; the immune lineages
# use hues that do not collide with them), so points of different lineages never share a colour
_lin_all = list(dict.fromkeys(CT_LEVELS['taa_l1'].dropna()))
LINEAGE_COLORS = {'TEC': '#0072B2', 'Fb': '#E69F00', 'EC': '#009E73', 'Mural': '#CC79A7',
                  'T': '#D62728', 'NK': '#9467BD', 'B': '#8C564B', 'Mono': '#7F7F7F',
                  'Mac': '#BCBD22', 'DC': '#17BECF', 'Myeloid_dev': '#000000',
                  'Adipocyte': '#FF9896', 'Neutrophil': '#C5B0D5', 'Mast': '#C49C94'}
_extra = sns.color_palette('Set2', n_colors=8)                    # fallback for any other lineage
for i, l in enumerate([l for l in _lin_all if l not in LINEAGE_COLORS]):
    LINEAGE_COLORS[l] = matplotlib.colors.to_hex(_extra[i % len(_extra)])
print(f'{len(_lin_all)} lineages (taa_l1): {", ".join(_lin_all)}')
print(f'{CT_LEVELS[LEVEL].nunique()} cell types at {LEVEL}; stroma = {ct_order(lineages=STROMA_L1)}')


14 lineages (taa_l1): T, NK, TEC, Fb, Adipocyte, EC, Mural, B, Myeloid_dev, Neutrophil, Mono, Mac, DC, Mast
30 cell types at taa_l2_v2; stroma = ['cTEC', 'mcTEC', 'mTEC', 'TEC-EMT', 'Fb', 'EC-blood', 'EC-lymphatic', 'SMC', 'Pericyte']


## 1. The two stromal rankings

Each ranking is the median within-cell-type percentile of a regulon over the 17 stromal population ×
contrast slots (notebooks 01 and 02, section 6): CollecTRI percentiles are among all TFs, SCENIC+
percentiles within the eRegulon's class. `kept` marks regulons with at least 15 measured targets.


In [5]:
ct_rank = pd.read_csv(f'{RESULTS}/{CT_PREFIX}_sharedRanking_stroma.csv', index_col=0)
ct_top  = pd.read_csv(f'{RESULTS}/{CT_PREFIX}_top20_sharedStroma.csv', index_col=0)
sc_rank = pd.read_csv(f'{RESULTS}/{SC_PREFIX}_sharedRanking_stroma.csv', index_col=0)
sc_top  = pd.read_csv(f'{RESULTS}/{SC_PREFIX}_top20_sharedStroma.csv', index_col=0)
ct_rank['tf'], ct_top['tf'] = ct_rank.index, ct_top.index
for name, r, t in (('CollecTRI', ct_rank, ct_top), ('SCENIC+', sc_rank, sc_top)):
    print(f'{name}: {len(r)} regulons ranked ({int(r["kept"].sum())} with >= 15 targets), '
          f'{r["tf"].nunique()} TFs; top-{N_TOP} lists: '
          f'{t["tf"][t["direction"] == "up"].nunique()} TFs up, {t["tf"][t["direction"] == "down"].nunique()} TFs down')
display(pd.concat({'CollecTRI': ct_top.head(3), 'SCENIC+': sc_top.head(3)}))


CollecTRI: 677 regulons ranked (467 with >= 15 targets), 677 TFs; top-20 lists: 20 TFs up, 20 TFs down
SCENIC+: 395 regulons ranked (356 with >= 15 targets), 294 TFs; top-20 lists: 19 TFs up, 20 TFs down


median_pct     mean_z    mean_r  n_slots  n_sig  n_targets  kept direction     tf        mode
          source                                                                                                          
CollecTRI BMAL1               99.580420   4.637302  0.031586       17     16         57  True        up  BMAL1         NaN
          STAT3               99.131693   3.973484  0.028898       17     14        429  True        up  STAT3         NaN
          NR3C1               98.331015   3.412163  0.024350       17     11        288  True        up  NR3C1         NaN
SCENIC+   KLF9_+_+_(1063g)    99.300699  16.431580  0.119265       17     17       1063  True        up   KLF9  activating
          NFIL3_+_+_(653g)    98.251748  14.020689  0.103010       17     16        653  True        up  NFIL3  activating
          FOSL2_+_+_(1048g)   97.552448  14.289380  0.105695       17     17       1048  True        up  FOSL2  activating

## 2. TF families

Families are assigned by name with the regular expressions below (first match wins; anything
unmatched is `other`). They are deliberately coarse: the question is whether the two networks
nominate the same *programmes*, not the same members. Extend the list to taste, the notebook does
not depend on it being complete.


In [6]:
TF_FAMILIES = [
    ('CREB / ATF1',            r'^(CREB\d|CREM|ATF1)$'),
    ('AP-1 / bZIP',            r'^(JUN|JUNB|JUND|FOS|FOSB|FOSL\d|ATF\d|BATF\d?|MAF[ABFGK]?|BACH\d|JDP2|NFE2)$'),
    ('CEBP',                   r'^CEBP[ABDGEZ]$'),
    ('KLF / SP',               r'^(KLF\d+|SP[1-9])$'),
    ('NFI',                    r'^NFI[ABCX]$'),
    ('NFIL3',                  r'^NFIL3$'),
    ('E2F / proliferation',    r'^(E2F\d|TFDP\d|MYBL\d|MAZ|ZNF367|FOXM1|MCM\d|TP73)$'),
    ('IRF',                    r'^IRF\d$'),
    ('STAT',                   r'^STAT\d[AB]?$'),
    ('NF-kB',                  r'^(NFKB\d?|REL|RELA|RELB)$'),
    ('RFX / MHC-II',           r'^(RFX\w*|CIITA)$'),
    ('SREBF / lipid',          r'^(SREBF\d|PPARG|MLXIPL|NR1H[234])$'),
    ('NRF2 / oxidative',       r'^(NFE2L\d)$'),
    ('HIF / hypoxia',          r'^(HIF1A|EPAS1|ARNT)$'),
    ('p53',                    r'^(TP53|TP63)$'),
    ('steroid NR',             r'^(AR|ESR\d|PGR|NR3C\d)$'),
    ('circadian',              r'^(CLOCK|BMAL1|ARNTL\d?|NPAS2|PER\d|CRY\d|NR1D\d|DBP|TEF|HLF|BHLHE4\d|CIART)$'),
    ('nuclear receptor (other)', r'^(NR\d[A-Z]\d|RAR[ABG]|RXR[ABG]|THR[AB]|VDR|PPAR[AD]|ROR[ABC]|HNF4[AG]|ESRR[ABG]|NCOA\d)$'),
    ('FOXO',                   r'^FOXO\d$'),
    ('FOX (other)',            r'^FOX[A-Z]\d?$'),
    ('SMAD',                   r'^SMAD\d$'),
    ('MEF2',                   r'^MEF2[ABCD]$'),
    ('MYC / MAX',              r'^(MYC|MYCN|MAX|MXI1|MNT|MXD\d)$'),
    ('lymphoid',               r'^(IKZF\d|BCL11[AB]|LEF1|TCF7|POU2F2|PAX5|BCL6|EBF\d|SATB1)$'),
    ('ETS',                    r'^(ERG|FLI1|ETS\d|ETV\d|ELK\d|ELF\d|SPI1|SPIB)$'),
    ('chromatin / HDAC',       r'^(HDAC\d+|MECP2|EZH2|CTCF|KDM\w+|SMARC\w+|ARID\w+)$'),
    ('GLI / Hedgehog',         r'^GLI\d$'),
    ('TEAD / Hippo',           r'^(TEAD\d|YAP1|WWTR1)$'),
    ('homeobox / developmental', r'^(CDX\d|HOX\w+|PBX\d|MEIS\d|NKX\d-\d|PAX\d|SIX\d|TBX\d+|HES\d|HEY\d|SOX\d+|GATA\d|HAND\d|TWIST\d|PRRX\d|LHX\d|MSX\d|DLX\d)$'),
    ('ZBTB',                   r'^ZBTB\d+$'),
    ('ZNF',                    r'^ZNF\d+$'),
]
FAMILY_ORDER = [f for f, _ in TF_FAMILIES] + ['other']


def tf_family(tf):
    for name, rx in TF_FAMILIES:
        if re.match(rx, str(tf)):
            return name
    return 'other'


ct_rank['family'] = ct_rank['tf'].map(tf_family)
sc_rank['family'] = sc_rank['tf'].map(tf_family)
ct_top['family'], sc_top['family'] = ct_top['tf'].map(tf_family), sc_top['tf'].map(tf_family)
sizes = pd.concat({'CollecTRI': ct_rank[ct_rank['kept']].drop_duplicates('tf')['family'].value_counts(),
                   'SCENIC+': sc_rank[sc_rank['kept']].drop_duplicates('tf')['family'].value_counts()}, axis=1).fillna(0).astype(int)
print('TFs per family among the ranked regulons (>= 15 targets):')
display(sizes.reindex([f for f in FAMILY_ORDER if f in sizes.index]).T)


TFs per family among the ranked regulons (>= 15 targets):


family,CREB / ATF1,AP-1 / bZIP,CEBP,KLF / SP,NFI,NFIL3,E2F / proliferation,IRF,STAT,NF-kB,RFX / MHC-II,SREBF / lipid,NRF2 / oxidative,HIF / hypoxia,p53,steroid NR,circadian,nuclear receptor (other),FOXO,FOX (other),SMAD,MEF2,MYC / MAX,lymphoid,ETS,chromatin / HDAC,GLI / Hedgehog,TEAD / Hippo,homeobox / developmental,ZBTB,ZNF,other
CollecTRI,4,20,6,16,4,1,10,9,7,6,5,7,1,3,2,6,8,34,3,12,7,3,4,10,15,11,3,1,56,2,7,184
SCENIC+,2,16,2,12,4,1,11,5,5,5,2,3,3,2,1,2,3,13,2,5,4,3,2,13,13,3,1,3,37,5,12,74


## 3. Families in the top-20 lists

Distinct TFs per family in each network's top-20 up and top-20 down lists (a TF contributing two
SCENIC+ eRegulons counts once), with the members. `in_both` marks families that both networks
nominate in the same direction.


In [7]:
rows = []
for d in ('up', 'down'):
    for net, top in (('CollecTRI', ct_top), ('SCENIC+', sc_top)):
        for tf in dict.fromkeys(top.loc[top['direction'] == d, 'tf']):
            rows.append({'direction': d, 'network': net, 'tf': tf, 'family': tf_family(tf)})
top_long = pd.DataFrame(rows)
top_long.to_csv(f'{RESULTS}/{PREFIX}_top20_members.csv', index=False)

counts = (top_long.groupby(['direction', 'family', 'network'])['tf'].nunique()
          .unstack('network').fillna(0).astype(int))
counts['in_both'] = (counts[['CollecTRI', 'SCENIC+']] > 0).all(axis=1)
members = (top_long.groupby(['direction', 'family', 'network'])['tf']
           .apply(lambda s: ', '.join(sorted(set(s)))).unstack('network').fillna(''))
fam_tab = counts.join(members, rsuffix=' TFs').rename(columns={'CollecTRI TFs': 'CollecTRI members',
                                                                'SCENIC+ TFs': 'SCENIC+ members'})
fam_tab.to_csv(f'{RESULTS}/{PREFIX}_top20_families.csv')
for d in ('up', 'down'):
    t = fam_tab.loc[d]
    t = t.assign(total=t['CollecTRI'] + t['SCENIC+']).sort_values(['in_both', 'total'], ascending=False).drop(columns='total')
    print(f'\n{d.upper()} with age - families in the top-{N_TOP} lists ({int(t["in_both"].sum())} nominated by both networks):')
    display(t)

# figure: dodged horizontal bars per family, one panel per direction
fig = mm_fig(NATURE_W['two-thirds'], 95)
axes = fig.subplots(1, 2, sharex=True)
for ax, d in zip(axes, ('up', 'down')):
    t = counts.loc[d].copy()
    t['total'] = t['CollecTRI'] + t['SCENIC+']
    t = t.sort_values(['in_both', 'total'], ascending=True)
    y = np.arange(len(t))
    for i, (net, col) in enumerate(NET_COLORS.items()):
        ax.barh(y + (i - 0.5) * 0.38, t[net], height=0.36, color=col, label=net, linewidth=0)
    ax.set_yticks(y)
    ax.set_yticklabels(t.index)
    ax.set_title(f'{d} with age (top {N_TOP})')
    ax.set_xlabel('TFs in the list')
    ax.grid(axis='x', color='0.92', lw=0.4)
    ax.set_axisbelow(True)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
axes[0].legend(frameon=False, loc='lower right')
save_fig(fig, f'{PREFIX}_top20_families')
plt.show()



UP with age - families in the top-20 lists (5 nominated by both networks):


network,CollecTRI,SCENIC+,in_both,CollecTRI members,SCENIC+ members
family,,,,,
AP-1 / bZIP,2,4,True,"JUN, JUND","ATF3, BACH1, FOSL2, MAFF"
KLF / SP,1,3,True,KLF4,"KLF10, KLF4, KLF9"
CEBP,1,2,True,CEBPB,"CEBPB, CEBPD"
STAT,2,1,True,"STAT3, STAT5B",STAT3
MYC / MAX,1,1,True,MAX,MYC
steroid NR,3,0,False,"AR, NR3C1, PGR",
FOXO,2,0,False,"FOXO1, FOXO3",
NFI,0,2,False,,"NFIA, NFIB"
circadian,2,0,False,"BMAL1, CLOCK",



DOWN with age - families in the top-20 lists (4 nominated by both networks):


network,CollecTRI,SCENIC+,in_both,CollecTRI members,SCENIC+ members
family,,,,,
E2F / proliferation,1,7,True,E2F4,"E2F1, E2F2, E2F7, E2F8, MAZ, MYBL2, ZNF367"
IRF,6,2,True,"IRF1, IRF2, IRF3, IRF4, IRF7, IRF9","IRF7, IRF8"
lymphoid,3,4,True,"BCL6, POU2F2, SATB1","BCL11A, BCL11B, IKZF1, IKZF3"
SREBF / lipid,1,1,True,SREBF2,SREBF1
RFX / MHC-II,3,0,False,"RFX1, RFXANK, RFXAP",
chromatin / HDAC,3,0,False,"HDAC1, HDAC9, MECP2",
ETS,0,2,False,,"ERG, FLI1"
STAT,0,2,False,,"STAT1, STAT2"
AP-1 / bZIP,1,0,False,JDP2,


saved thyAgeing_TFfamilies_ageDE_taa_l2_v2_top20_families.pdf  (120 x 95 mm)


## 4. Whole families across the two networks

The top-20 cut hides most of a family. Here every ranked TF is placed by its median stromal
percentile (CollecTRI: among all TFs; SCENIC+: the TF's activating eRegulon, largest if several,
within its class), grouped by family, for the families with at least three TFs in one of the
networks. A family sitting high in both panels is a shared programme; one high in only one network
is a property of that network's regulon definitions.


In [8]:
sc_act = (sc_rank[sc_rank['kept'] & (sc_rank['mode'] == 'activating')]
          .sort_values('n_targets', ascending=False).drop_duplicates('tf'))
long = pd.concat([ct_rank[ct_rank['kept']].assign(network='CollecTRI')[['tf', 'family', 'network', 'median_pct', 'n_targets', 'n_sig']],
                  sc_act.assign(network='SCENIC+')[['tf', 'family', 'network', 'median_pct', 'n_targets', 'n_sig']]],
                 ignore_index=True)
long.to_csv(f'{RESULTS}/{PREFIX}_rankedTFs_byFamily.csv', index=False)

fam = long.groupby(['family', 'network'])['median_pct'].agg(n='size', median='median').unstack('network')
show = [f for f in FAMILY_ORDER if f in fam.index and f != 'other'
        and (fam.loc[f, ('n', 'CollecTRI')] >= 3 or fam.loc[f, ('n', 'SCENIC+')] >= 3)]
fam_show = fam.loc[show].copy()
fam_show['mean_of_medians'] = fam_show['median'].mean(axis=1)
fam_show = fam_show.sort_values('mean_of_medians', ascending=False)
fam_show.to_csv(f'{RESULTS}/{PREFIX}_families_medianPercentile.csv')
print('families with >= 3 TFs in a network - number of TFs and median of the TF percentiles:')
display(fam_show.round(1))

# figure: families on y (ordered), one strip per network
order = list(fam_show.index)
fig = mm_fig(NATURE_W['two-thirds'], 4.6 * len(order) + 22)
ax = fig.subplots()
for i, f in enumerate(order):
    for j, (net, col) in enumerate(NET_COLORS.items()):
        v = long.loc[(long['family'] == f) & (long['network'] == net), 'median_pct'].to_numpy()
        if not v.size:
            continue
        yy = i + (j - 0.5) * 0.36
        ax.scatter(v, np.full(v.size, yy), s=9, color=col, alpha=0.75, edgecolor='white', linewidths=0.3, zorder=3)
        ax.plot([np.median(v)] * 2, [yy - 0.16, yy + 0.16], color='k', lw=1.0, zorder=4)
ax.axvline(50, color='0.6', lw=0.5, ls=':')
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f'{f}  ({int(fam_show.loc[f, ("n", "CollecTRI")])} / {int(fam_show.loc[f, ("n", "SCENIC+")])})' for f in order])
ax.set_ylim(len(order) - 0.4, -0.6)
ax.set_xlim(-2, 102)
ax.set_xlabel('median percentile across the stromal population x contrast slots')
ax.grid(axis='x', color='0.92', lw=0.4)
ax.set_axisbelow(True)
handles = [Line2D([], [], color=c, marker='o', linestyle='none', markersize=4, label=n) for n, c in NET_COLORS.items()]
handles += [Line2D([], [], color='k', lw=1.0, label='family median')]
ax.legend(handles=handles, frameon=False, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3)
ax.set_title('TF families (n TFs: CollecTRI / SCENIC+)', pad=14)
save_fig(fig, f'{PREFIX}_families_percentiles')
plt.show()


families with >= 3 TFs in a network - number of TFs and median of the TF percentiles:


n            median         mean_of_medians
network                  CollecTRI SCENIC+ CollecTRI SCENIC+                
family                                                                      
CEBP                             6       2      86.1    95.6            90.8
FOXO                             3       2      94.0    78.1            86.1
steroid NR                       6       1      91.9    73.7            82.8
HIF / hypoxia                    3       2      84.3    71.1            77.7
CREB / ATF1                      4       2      67.5    80.8            74.2
AP-1 / bZIP                     20      15      59.8    87.1            73.4
TEAD / Hippo                     1       3      59.3    77.5            68.4
NFI                              4       4      47.0    86.4            66.7
KLF / SP                        16      12      54.3    76.2            65.3
nuclear receptor (other)        34      12      68.0    56.0            62.0
SREBF / lipid                    7       3      62.4    61.4            61.9
NRF2 / oxidative                 1       3      90.5    28.7            59.6
ETS                             15      13      60.0    56.8            58.4
NF-kB                            6       5      51.3    64.2            57.7
MYC / MAX                        4       2      59.4    53.7            56.5
MEF2                             3       3      58.4    53.8            56.1
ZBTB                             2       5      34.9    73.7            54.3
SMAD                             7       4      58.4    48.1            53.2
STAT                             7       5      63.1    42.2            52.7
FOX (other)                     12       4      50.4    52.6            51.5
GLI / Hedgehog                   3       1      29.8    68.1            48.9
homeobox / developmental        56      36      47.8    44.7            46.3
ZNF                              7      12      47.6    41.2            44.4
circadian                        8       3      56.6    31.5            44.0
chromatin / HDAC                11       3      41.4    45.1            43.3
RFX / MHC-II                     5       1       9.4    70.9            40.1
E2F / proliferation             10      10      53.9     8.0            31.0
lymphoid                        10      13      25.5    28.1            26.8
IRF                              9       5       8.6    25.3            16.9

saved thyAgeing_TFfamilies_ageDE_taa_l2_v2_families_percentiles.pdf  (120 x 155 mm)


In [9]:
# the same family view split by contrast, from the per-cell-type score tables
ct_scores = pd.read_csv(f'{RESULTS}/{CT_PREFIX}_scores_perCellType.csv', usecols=['cell_type', 'contrast', 'source', 'pct_z'])
sc_scores = pd.read_csv(f'{RESULTS}/{SC_PREFIX}_scores_perCellType.csv', usecols=['cell_type', 'contrast', 'source', 'tf', 'mode', 'pct_z_mode'])
stroma_ct = ct_order(lineages=STROMA_L1, present=ct_scores['cell_type'])
ct_c = (ct_scores[ct_scores['cell_type'].isin(stroma_ct) & ct_scores['source'].isin(long.loc[long['network'] == 'CollecTRI', 'tf'])]
        .groupby(['source', 'contrast'])['pct_z'].median().rename('median_pct').reset_index()
        .rename(columns={'source': 'tf'}).assign(network='CollecTRI'))
sc_c = (sc_scores[sc_scores['cell_type'].isin(stroma_ct) & sc_scores['source'].isin(sc_act.index)]
        .groupby(['tf', 'contrast'])['pct_z_mode'].median().rename('median_pct').reset_index().assign(network='SCENIC+'))
by_c = pd.concat([ct_c, sc_c], ignore_index=True)
by_c['family'] = by_c['tf'].map(tf_family)
fam_c = (by_c[by_c['family'].isin(order)].groupby(['family', 'network', 'contrast'])['median_pct'].median()
         .unstack(['network', 'contrast']).reindex(order))
fam_c.to_csv(f'{RESULTS}/{PREFIX}_families_medianPercentile_byContrast.csv')
print('family median percentile by contrast (stroma):')
display(fam_c.round(1))


family median percentile by contrast (stroma):


network                        CollecTRI                        SCENIC+               
contrast                 adult_vs_infant paed_vs_infant adult_vs_infant paed_vs_infant
family                                                                                
CEBP                                90.8           80.9            97.9           77.5
FOXO                                96.3           88.7            74.8           82.1
steroid NR                          92.6           82.2            76.5           64.6
HIF / hypoxia                       85.6           69.5            77.2           62.6
CREB / ATF1                         76.1           59.4            79.7           71.6
AP-1 / bZIP                         59.0           59.6            85.7           87.7
TEAD / Hippo                        60.6           51.9            78.7           68.3
NFI                                 47.0           42.8            91.1           69.7
KLF / SP                            53.1           53.2            78.3           78.4
nuclear receptor (other)            66.8           66.8            59.6           43.2
SREBF / lipid                       74.0           41.0            52.4           49.1
NRF2 / oxidative                    98.4           59.6            69.8           44.3
ETS                                 61.5           52.1            62.8           34.8
NF-kB                               41.9           65.7            61.5           83.9
MYC / MAX                           63.6           45.7            57.0           46.4
MEF2                                67.7           59.7            52.9           50.2
ZBTB                                36.7           35.7            62.6           78.5
SMAD                                67.5           54.2            51.4           44.4
STAT                                75.9           67.3            43.4           41.3
FOX (other)                         46.9           47.6            43.5           53.5
GLI / Hedgehog                      34.8           28.2            73.4           58.9
homeobox / developmental            41.1           51.4            45.8           43.0
ZNF                                 38.9           52.9            44.8           35.8
circadian                           54.1           60.8            26.7           34.8
chromatin / HDAC                    37.6           40.4            43.4           46.0
RFX / MHC-II                        13.2            8.3            74.1           68.0
E2F / proliferation                 56.6           48.1             7.7            7.4
lymphoid                            30.6           22.5            15.4           53.8
IRF                                  6.6           34.1            10.5           45.1

## 5. TF-level agreement for the TFs present in both networks

For each TF with a CollecTRI regulon and an activating SCENIC+ eRegulon, the two median stromal
percentiles are compared. Agreement is expected to be partial: the two regulons of a TF share few
targets (section 6 shows this for AR and NR3C1), so disagreement here is mostly a statement about
the networks, not about the scoring.


In [10]:
both = (long[long['network'] == 'CollecTRI'][['tf', 'family', 'median_pct', 'n_targets']]
        .merge(long[long['network'] == 'SCENIC+'][['tf', 'median_pct', 'n_targets']], on='tf', suffixes=('_CollecTRI', '_SCENIC')))
rho, p = spearmanr(both['median_pct_CollecTRI'], both['median_pct_SCENIC'])
up_ct, dn_ct = set(ct_top.loc[ct_top['direction'] == 'up', 'tf']), set(ct_top.loc[ct_top['direction'] == 'down', 'tf'])
up_sc, dn_sc = set(sc_top.loc[sc_top['direction'] == 'up', 'tf']), set(sc_top.loc[sc_top['direction'] == 'down', 'tf'])
both['top20'] = np.select([both['tf'].isin(up_ct & up_sc), both['tf'].isin(dn_ct & dn_sc),
                           both['tf'].isin(up_ct | up_sc), both['tf'].isin(dn_ct | dn_sc)],
                          ['up in both', 'down in both', 'up in one', 'down in one'], default='')
both = both.sort_values('median_pct_CollecTRI', ascending=False)
both.to_csv(f'{RESULTS}/{PREFIX}_TFagreement.csv', index=False)
print(f'{len(both)} TFs ranked in both networks: Spearman rho = {rho:.2f} (p = {p:.1e})')
print('up in both top-20 lists:  ', sorted(up_ct & up_sc))
print('down in both top-20 lists:', sorted(dn_ct & dn_sc))
display(both[both['top20'].str.endswith('both')].round(1))

# scatter, coloured by the families both networks nominate most
hl = [f for f in fam_show.index[:8]] + ['steroid NR']
hl = list(dict.fromkeys(f for f in hl if f != 'other'))
pal = dict(zip(hl, sns.color_palette('tab10', n_colors=len(hl))))
fig = mm_fig(NATURE_W['single'], 100)
ax = fig.subplots()
other = ~both['family'].isin(hl)
ax.scatter(both.loc[other, 'median_pct_CollecTRI'], both.loc[other, 'median_pct_SCENIC'], s=7, color='0.75',
           linewidths=0, zorder=2, label='other')
for f in hl:
    m = both['family'] == f
    ax.scatter(both.loc[m, 'median_pct_CollecTRI'], both.loc[m, 'median_pct_SCENIC'], s=12, color=pal[f],
               edgecolor='white', linewidths=0.3, zorder=3, label=f)
lab = both[both['top20'].str.endswith('both') | both['tf'].isin(STEROID_TFS)]
for _, r in lab.iterrows():
    ax.annotate(r['tf'], (r['median_pct_CollecTRI'], r['median_pct_SCENIC']), fontsize=5.5,
                xytext=(2, 2), textcoords='offset points')
ax.plot([0, 100], [0, 100], color='0.6', lw=0.5, ls=':')
ax.set_xlim(-2, 102)
ax.set_ylim(-2, 102)
ax.set_xlabel('CollecTRI: median percentile (stroma)')
ax.set_ylabel('SCENIC+ activating eRegulon: median percentile within class (stroma)')
ax.set_title(f'{len(both)} TFs in both networks, Spearman rho = {rho:.2f}')
ax.legend(frameon=False, fontsize=5.5, loc='upper left', bbox_to_anchor=(1.01, 1.0))
save_fig(fig, f'{PREFIX}_TFagreement_scatter')
plt.show()


166 TFs ranked in both networks: Spearman rho = 0.28 (p = 2.2e-04)
up in both top-20 lists:   ['CEBPB', 'KLF4', 'STAT3']
down in both top-20 lists: ['IRF7']


,tf,family,median_pct_CollecTRI,n_targets_CollecTRI,median_pct_SCENIC,n_targets_SCENIC,top20
0,STAT3,STAT,99.1,429,92.7,336,up in both
5,KLF4,KLF / SP,94.3,129,96.1,794,up in both
6,CEBPB,CEBP,94.3,441,94.0,467,up in both
162,IRF7,IRF,8.6,24,6.6,27,down in both


saved thyAgeing_TFfamilies_ageDE_taa_l2_v2_TFagreement_scatter.pdf  (89 x 100 mm)


## 6. AR and NR3C1: the CollecTRI regulon versus the SCENIC+ eRegulon

The two networks define very different target sets for the same receptor. CollecTRI collects
literature-reported targets across tissues (signed: +1 activation, −1 repression); SCENIC+ infers
targets from enhancer accessibility and co-expression in the thymus itself, and gives each TF one
activating and/or one repressing eRegulon. This section measures the overlap (with a hypergeometric
test against the genes both networks could have chosen from), checks whether shared targets carry
the same sign, and then asks how the shared, CollecTRI-only and SCENIC+-only targets behave in the
stromal age contrasts — which is what decides whether the two networks are reporting the same
biology for these receptors.


In [11]:
DE_COLS = ['cell_type', 'gene_name', 'baseMean', 'stat']


def read_de_table(path):
    """One contrast: flat `cell_type, gene_name, baseMean, stat`. The gene column is `gene_name` or `gene`."""
    head = pd.read_csv(path, nrows=0).columns
    gene_col = 'gene_name' if 'gene_name' in head else 'gene'
    return (pd.read_csv(path, usecols=['cell_type', gene_col, 'baseMean', 'stat'])
              .rename(columns={gene_col: 'gene_name'})[DE_COLS])


def load_de(level, contrasts, dea_dir=DEA_DIR, stat_cap=STAT_CAP):
    """Both contrasts in one long table, Wald statistic capped at +/- stat_cap."""
    de = pd.concat([read_de_table(f'{dea_dir}/thyAgeing_dea_{level}_{c}_ageEffect.csv').assign(contrast=c)
                    for c in contrasts], ignore_index=True)
    assert np.isfinite(de['stat']).all(), 'non-finite Wald statistics in the DE tables'
    hit = de.loc[de['stat'].abs() > stat_cap, ['cell_type', 'contrast', 'gene_name', 'baseMean', 'stat']]
    print(f'{len(hit)} of {len(de):,} Wald statistics capped at +/-{stat_cap:g}')
    if len(hit):
        display(hit)
    de['stat'] = de['stat'].clip(-stat_cap, stat_cap)
    return de


de = load_de(LEVEL, CONTRASTS)

ct_by_contrast = {c: sorted(de.loc[de['contrast'] == c, 'cell_type'].unique()) for c in CONTRASTS}
shared_cts = sorted(set.intersection(*map(set, ct_by_contrast.values())))
for c, v in ct_by_contrast.items():
    extra = sorted(set(v) - set(shared_cts))
    print(f'{c}: {len(v)} cell types' + (f'  (only in this contrast: {", ".join(extra)})' if extra else ''))
n_by_ct = de.groupby('cell_type')['gene_name'].nunique()
print(f'{len(n_by_ct)} cell types in total, tested in both contrasts: {len(shared_cts)}')
print(f'genes tested per cell type: {n_by_ct.min():,} - {n_by_ct.max():,} (median {int(n_by_ct.median()):,})')
display(n_by_ct.reindex(ct_order(present=n_by_ct.index)).to_frame('genes tested').T)


2 of 870,930 Wald statistics capped at +/-20


,cell_type,contrast,gene_name,baseMean,stat
629836,TEC-EMT,adult_vs_infant,SCGB2A2,6.103285,3.361936e+07
632220,TEC-EMT,adult_vs_infant,IGHV2-5,6.669867,5.170339e+04


paed_vs_infant: 23 cell types
adult_vs_infant: 26 cell types  (only in this contrast: EC-lymphatic, Mono, Myeloid_dev)


26 cell types in total, tested in both contrasts: 23
genes tested per cell type: 8,722 - 27,510 (median 17,585)


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
genes tested,15279,20652,23382,18467,20913,20087,18483,13505,20012,21699,27510,17274,23646,22126,13490,17897,15498,20005,15027,8722,11482,14048,13352,12979,13273,13504


In [12]:
from scipy.stats import hypergeom

ct_net = pd.read_csv(f'{DATA_DIR}/CollecTRI_regulons_human.csv')
sc_net = pd.read_csv(f'{DATA_DIR}/ThyAgeing_SCENICplus_eRegulons_decoupler_net_signonly.csv')
_SRC_RE = re.compile(r'^(.+)_([+-])_([+-])_\((\d+)[gr]\)$')
sc_tf_of = {s: _SRC_RE.match(s).group(1) for s in sc_net['source'].unique()}
universe = set(ct_net['target']) | set(sc_net['target'])          # genes either network could have chosen
pool_ct, pool_sc = set(ct_net['target']), set(sc_net['target'])
N_common = len(pool_ct & pool_sc)                                  # genes present in both networks' target pools

TARGET_SETS = {}
rows = []
for tf in ['AR', 'NR3C1']:
    ct_t = ct_net[ct_net['source'] == tf].drop_duplicates('target').set_index('target')['weight']
    for reg in sorted(s for s, t in sc_tf_of.items() if t == tf):
        sc_t = sc_net[sc_net['source'] == reg].drop_duplicates('target').set_index('target')['weight']
        shared = ct_t.index.intersection(sc_t.index)
        # hypergeometric: draws = SCENIC targets that lie in CollecTRI's pool, successes = CollecTRI targets in SCENIC's pool
        k, K, n = len(shared), len(set(ct_t.index) & pool_sc), len(set(sc_t.index) & pool_ct)
        p_hyper = hypergeom.sf(k - 1, N_common, K, n)
        same = int((np.sign(ct_t[shared]) == np.sign(sc_t[shared])).sum())
        TARGET_SETS[(tf, reg)] = (ct_t, sc_t, list(shared))
        rows.append({'TF': tf, 'SCENIC+ eRegulon': reg, 'CollecTRI targets': len(ct_t), 'SCENIC+ targets': len(sc_t),
                     'shared': k, 'expected by chance': round(K * n / N_common, 1), 'p (hypergeometric)': p_hyper,
                     'overlap coefficient': k / min(len(ct_t), len(sc_t)), 'Jaccard': k / len(set(ct_t.index) | set(sc_t.index)),
                     'same sign in both': same,
                     'CollecTRI sign of shared targets': ', '.join(f'{g}({"+" if ct_t[g] > 0 else "-"})' for g in shared)})
overlap = pd.DataFrame(rows)
overlap.to_csv(f'{RESULTS}/{PREFIX}_AR_NR3C1_targetOverlap.csv', index=False)
print(f'{N_common:,} genes are targets in both networks (the background of the hypergeometric test)')
display(overlap.drop(columns='CollecTRI sign of shared targets').round(3))
for _, r in overlap.iterrows():
    print(f"\n{r['TF']} / {r['SCENIC+ eRegulon']} - shared targets with their CollecTRI sign: {r['CollecTRI sign of shared targets'] or 'none'}")


4,486 genes are targets in both networks (the background of the hypergeometric test)


,TF,SCENIC+ eRegulon,CollecTRI targets,SCENIC+ targets,shared,expected by chance,p (hypergeometric),overlap coefficient,Jaccard,same sign in both
0,AR,AR_+_+_(78g),514,78,7,4.2,0.119,0.090,0.012,6
1,NR3C1,NR3C1_-_+_(18g),292,18,3,0.5,0.013,0.167,0.010,2



AR / AR_+_+_(78g) - shared targets with their CollecTRI sign: AR(+), GREB1(+), EGFR(-), IGF1(+), SREBF1(+), SNAI2(+), GLRX(+)

NR3C1 / NR3C1_-_+_(18g) - shared targets with their CollecTRI sign: JUN(-), FOS(-), DUSP1(+)


### How the target classes move with age in the stroma

For each receptor, targets are split into *shared*, *CollecTRI only* and *SCENIC+ only*, and their
Wald statistics summarised over the stromal populations. The heatmaps then show every SCENIC+
eRegulon target in every stromal population (adult vs infant), with the CollecTRI sign appended
where the gene is also a CollecTRI target. For the repressing NR3C1 eRegulon, remember that
notebook 02 scores it as *activity up* when these targets go *down*.


In [13]:
# `de` was loaded by the shared DE cell above
stroma_ct = ct_order(lineages=STROMA_L1, present=de['cell_type'])
de_st = de[de['cell_type'].isin(stroma_ct)]

rows = []
for (tf, reg), (ct_t, sc_t, shared) in TARGET_SETS.items():
    classes = {'shared': set(shared), 'CollecTRI only': set(ct_t.index) - set(sc_t.index),
               'SCENIC+ only': set(sc_t.index) - set(ct_t.index)}
    for cls, genes in classes.items():
        for c in CONTRASTS:
            d = de_st[(de_st['contrast'] == c) & de_st['gene_name'].isin(genes)]
            per_gene = d.groupby('gene_name')['stat'].mean()          # mean over the stromal populations
            rows.append({'TF': tf, 'eRegulon': reg, 'class': cls, 'contrast': c, 'n_genes': len(genes),
                         'n_measured': len(per_gene), 'mean_stat': per_gene.mean(), 'median_stat': per_gene.median(),
                         'fraction_up': float((per_gene > 0).mean()) if len(per_gene) else np.nan})
cls_tab = pd.DataFrame(rows)
cls_tab.to_csv(f'{RESULTS}/{PREFIX}_AR_NR3C1_targetClasses_ageDE.csv', index=False)
print('per-gene Wald statistic (mean over the stromal populations), summarised per target class:')
display(cls_tab.pivot(index=['TF', 'eRegulon', 'class'], columns='contrast',
                      values=['n_measured', 'mean_stat', 'fraction_up']).round(2))


def plot_gene_heatmap(de_long, genes, cts, contrast, labels=None, width_mm=NATURE_W['single'], row_mm=2.6,
                      base_pt=6.5, qlim=0.98, title=None):
    """Genes x populations coloured by the Wald statistic, rows ordered by their mean."""
    d = de_long[(de_long['contrast'] == contrast) & de_long['gene_name'].isin(genes) & de_long['cell_type'].isin(cts)]
    M = d.pivot_table(index='gene_name', columns='cell_type', values='stat').reindex(columns=cts)
    M = M.loc[M.mean(axis=1).sort_values(ascending=False).index]
    v = M.to_numpy().ravel()
    v = v[np.isfinite(v)]
    vmax = max(float(np.nanquantile(np.abs(v), qlim)), 1e-9)
    lab_mm = 1.05 * max(len(str(c)) for c in cts) + 4.0
    with plt.rc_context({'font.size': base_pt, 'xtick.labelsize': base_pt, 'ytick.labelsize': base_pt}):
        fig = mm_fig(width_mm, len(M) * row_mm + lab_mm + 12.0)
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.05], wspace=0.05)
        ax = fig.add_subplot(gs[0, 0])
        sns.heatmap(M, ax=ax, cmap=DIVERGING, center=0, vmin=-vmax, vmax=vmax, cbar=False, linewidths=0,
                    yticklabels=[(labels or {}).get(g, g) for g in M.index])
        ax.grid(False)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
        ax.tick_params(length=1.2, width=0.4, pad=1.2)
        cax = fig.add_subplot(gs[0, 1])
        fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), DIVERGING), cax=cax, label='DESeq2 Wald statistic')
        cax.tick_params(length=1.5, width=0.4, pad=1.5)
        if title:
            fig.suptitle(title, wrap=True)
    return fig, M


for (tf, reg), (ct_t, sc_t, shared) in TARGET_SETS.items():
    labels = {g: f'{g}  [CollecTRI {"+" if ct_t[g] > 0 else "-"}]' if g in ct_t.index else g for g in sc_t.index}
    measured = [g for g in sc_t.index if g in set(de_st['gene_name'])]
    fig, M = plot_gene_heatmap(de_st, measured, stroma_ct, 'adult_vs_infant', labels=labels,
                               title=f'{reg}: all eRegulon targets, adult vs infant (stroma); [..] = also a CollecTRI target')
    save_fig(fig, f'{PREFIX}_targets_{reg.split("_")[0]}_SCENICplus_eRegulon_stroma_adult_vs_infant')
    plt.show()


per-gene Wald statistic (mean over the stromal populations), summarised per target class:


n_measured                      mean_stat                    fraction_up               
contrast                             adult_vs_infant paed_vs_infant adult_vs_infant paed_vs_infant adult_vs_infant paed_vs_infant
TF    eRegulon        class                                                                                                      
AR    AR_+_+_(78g)    CollecTRI only           499.0          499.0            0.01           0.10             0.5           0.52
                      SCENIC+ only              71.0           71.0            0.84           0.15             0.7           0.55
                      shared                     7.0            7.0            1.13           0.48             1.0           0.71
NR3C1 NR3C1_-_+_(18g) CollecTRI only           285.0          285.0            0.12           0.12             0.5           0.52
                      SCENIC+ only              15.0           15.0            1.12           0.64             0.6           0.80
                      shared                     3.0            3.0            1.34           2.14             1.0           1.00

saved thyAgeing_TFfamilies_ageDE_taa_l2_v2_targets_AR_SCENICplus_eRegulon_stroma_adult_vs_infant.pdf  (89 x 231 mm)


saved thyAgeing_TFfamilies_ageDE_taa_l2_v2_targets_NR3C1_SCENICplus_eRegulon_stroma_adult_vs_infant.pdf  (89 x 75 mm)


## Outputs

`results/` (prefix `thyAgeing_TFfamilies_ageDE_<level>_`): `top20_members.csv`, `top20_families.csv`,
`rankedTFs_byFamily.csv`, `families_medianPercentile.csv`, `families_medianPercentile_byContrast.csv`,
`TFagreement.csv`, `AR_NR3C1_targetOverlap.csv`, `AR_NR3C1_targetClasses_ageDE.csv`.

`figures/`: `top20_families.pdf`, `families_percentiles.pdf`, `TFagreement_scatter.pdf`,
`targets_AR_SCENICplus_eRegulon_stroma_adult_vs_infant.pdf`,
`targets_NR3C1_SCENICplus_eRegulon_stroma_adult_vs_infant.pdf`.
